# Budget-Constrained Treatment Decisioning

## Objective

Previously, I built customer-level features, examined treatment-assignment assumptions, trained uplift models, and evaluated their targeting rankings.

In this session, I translate those predictions into a practical decision: **which customers should receive marketing communication when there is a limited budget?**

The decision engine combines predicted uplift with three configurable business assumptions:

* Value per incremental conversion
* Cost per customer contact
* Total contact budget

I compare uplift-based targeting with random targeting, targeting customers who are most likely to purchase, contacting everyone, and contacting nobody.

**Interpretation:** X5 provides a binary post-communication outcome but does not provide a verified monetary value per conversion or contact cost. All monetary inputs in this notebook are hypothetical scenario parameters. Predicted uplift is also not an independently verified individual causal effect.


In [1]:
# ============================================================
# 1. Setup and model-ready data
#
# Snowflake/dbt remains the source of truth.
# Local Parquet files are reproducible modeling caches.
#
# Delete the scoring cache if its upstream dbt model changes.
# ============================================================

from pathlib import Path
from getpass import getpass
import os
import sys

import joblib
import numpy as np
import pandas as pd
import snowflake.connector

from dotenv import load_dotenv

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# Find the repository root whether VS Code starts the
# notebook from notebooks/ or from the repository root.
PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dbt" / "dbt_project.yml").exists()
)

sys.path.insert(0, str(PROJECT_ROOT))

from src.decisioning import (
    build_contact_policy,
    summarize_policy,
)


load_dotenv(
    PROJECT_ROOT / ".env"
)


TRAINING_CACHE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_uplift_training.parquet"
)

SCORING_CACHE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "mart_uplift_scoring.parquet"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "models"
    / "logistic_t_learner_full.joblib"
)


# The training cache was created in notebook 02.
# We intentionally reuse it instead of querying again.
if not TRAINING_CACHE.exists():

    raise FileNotFoundError(
        "Training cache not found. Run notebook 02 first."
    )


training_df = pd.read_parquet(
    TRAINING_CACHE
)

training_df.columns = (
    training_df.columns.str.lower()
)


# ------------------------------------------------------------
# Retrieve the unlabeled competition population only once.
# Do not store passwords in the notebook or repository.
# ------------------------------------------------------------

if SCORING_CACHE.exists():

    scoring_df = pd.read_parquet(
        SCORING_CACHE
    )

else:

    connection = None
    cursor = None

    try:

        connection = snowflake.connector.connect(
            account=os.environ["SNOWFLAKE_ACCOUNT"],
            user=os.environ["SNOWFLAKE_USER"],
            password=getpass("Snowflake password: "),
            authenticator="username_password_mfa",
            role="DBT_DEV_ROLE",
            warehouse="RETAIL_DEV_WH",
            database="RETAIL_GROWTH",
            schema="DEV_MARTS",
        )

        cursor = connection.cursor()

        cursor.execute(
            """
            SELECT *
            FROM RETAIL_GROWTH.DEV_MARTS.MART_UPLIFT_SCORING
            """
        )

        scoring_df = cursor.fetch_pandas_all()

    finally:

        if cursor is not None:
            cursor.close()

        if connection is not None:
            connection.close()

    scoring_df.columns = (
        scoring_df.columns.str.lower()
    )

    SCORING_CACHE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    scoring_df.to_parquet(
        SCORING_CACHE,
        index=False,
    )


scoring_df.columns = (
    scoring_df.columns.str.lower()
)


# ------------------------------------------------------------
# Validate the actual data loaded into Python.
# These checks complement, rather than replace, dbt tests.
# ------------------------------------------------------------

assert len(training_df) == 200_039
assert len(scoring_df) == 200_123

assert training_df["client_id"].is_unique
assert scoring_df["client_id"].is_unique

assert set(training_df["client_id"]).isdisjoint(
    set(scoring_df["client_id"])
)

assert {
    "treatment_flg",
    "target",
}.issubset(training_df.columns)

assert not {
    "treatment_flg",
    "target",
}.intersection(scoring_df.columns)


print(f"Training customers: {len(training_df):,}")
print(f"Scoring customers: {len(scoring_df):,}")
print("Train/scoring overlap: 0")

Training customers: 200,039
Scoring customers: 200,123
Train/scoring overlap: 0


## 2. Refit the provisional targeting model

The logistic T-learner was selected as the provisional policy model based on the exploratory model comparison in notebook 03.

Previously, I trained the models using approximately 75% of labeled customers and evaluated their predictions on the remaining development-validation customers.

For this deployment-style scoring exercise, I now refit the same model specification using **all 200,039 labeled customers**. This gives the model access to all available labeled training observations before scoring the separate, unlabeled competition population.

This full-data refit does not create a new independent performance estimate. The notebook 03 evaluation remains an exploratory assessment of models fitted on the earlier training split.

I keep the treatment and control outcome models separate so that each customer receives two predicted probabilities:

$$
\widehat{P}(Y=1\mid X,T=1)
$$

and

$$
\widehat{P}(Y=1\mid X,T=0).
$$

Their difference is the predicted uplift score.


In [2]:

# ============================================================
# 2. Full-data refit of the provisional logistic T-learner
#
# Do not include:
# - client_id
# - treatment_flg
# - target
# - previously generated propensity/uplift predictions
#
# Treatment and control models must have independent
# preprocessing pipelines.
# ============================================================

EXCLUDE_COLUMNS = {
    "client_id",
    "treatment_flg",
    "target",
}

BANNED_DERIVED_COLUMNS = {
    "propensity_score",
    "predicted_uplift",
    "uplift_decile",
    "p_treatment",
    "p_control",
}

feature_columns = [
    column
    for column in training_df.columns
    if column not in EXCLUDE_COLUMNS
]

assert not (
    set(feature_columns)
    & BANNED_DERIVED_COLUMNS
), "A derived modeling/evaluation column entered the features."


# The scoring population must contain every feature
# used to train the model.
missing_features = (
    set(feature_columns)
    - set(scoring_df.columns)
)

assert not missing_features, (
    f"Missing scoring features: {missing_features}"
)


categorical_features = [
    "gender",
]

numeric_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]


X_train = training_df[feature_columns]

treatment = (
    training_df["treatment_flg"]
    .astype(int)
)

target = (
    training_df["target"]
    .astype(int)
)


# ------------------------------------------------------------
# Each model receives a fresh preprocessing pipeline.
#
# Logistic regression benefits from scaling numeric inputs.
# One-hot encoding converts gender into numeric indicators.
# ------------------------------------------------------------

def make_logistic_pipeline():

    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_transformer,
                numeric_features,
            ),
            (
                "categorical",
                categorical_transformer,
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                ),
            ),
        ]
    )


treatment_model = make_logistic_pipeline()
control_model = make_logistic_pipeline()


# ------------------------------------------------------------
# Fit each model only on customers who actually received
# its corresponding treatment condition.
# ------------------------------------------------------------

treatment_mask = (
    treatment == 1
)

control_mask = (
    treatment == 0
)


treatment_model.fit(
    X_train.loc[treatment_mask],
    target.loc[treatment_mask],
)

control_model.fit(
    X_train.loc[control_mask],
    target.loc[control_mask],
)


# ------------------------------------------------------------
# Save the full-data model bundle for later MLflow/API work.
#
# Only load model artifacts you created and trust.
# joblib uses pickle-based serialization.
# ------------------------------------------------------------

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

model_bundle = {
    "treatment_model": treatment_model,
    "control_model": control_model,
    "feature_columns": feature_columns,
    "model_family": "logistic_t_learner",
    "training_customers": len(training_df),
    "feature_cutoff": "2019-03-19 00:00:00",
}

joblib.dump(
    model_bundle,
    MODEL_PATH,
)

print("Full-data T-learner fitted.")
print(f"Model artifact: {MODEL_PATH}")

Full-data T-learner fitted.
Model artifact: /Users/me/Downloads/retail-growth-decision-platform/data/models/logistic_t_learner_full.joblib


## 3. Generate uplift scores for the scoring population

The original X5 scoring population contains 200,123 customers without observed treatment assignments or target outcomes.

For each customer, I estimate two purchase probabilities using the fitted treatment and control models. Their difference becomes the predicted uplift:

$$
\widehat{\tau}(X)
=
\widehat{P}(Y=1\mid X,T=1)
-
\widehat{P}(Y=1\mid X,T=0).
$$

A positive score means the model predicts a higher purchase probability under treatment. A negative score means the model predicts a lower purchase probability under treatment.

These are model predictions rather than directly observed individual treatment effects. I cannot measure actual policy performance on this unlabeled population because its target outcomes are unavailable.


In [3]:

# ============================================================
# 3. Score the original X5 competition test population
#
# Predict BOTH treatment conditions for every customer.
# Their probability difference is the uplift score.
#
# Keep customer IDs alongside the predictions so they
# can be used by the downstream decision engine.
# ============================================================

X_scoring = scoring_df[
    feature_columns
]


p_treatment = (
    treatment_model.predict_proba(
        X_scoring
    )[:, 1]
)

p_control = (
    control_model.predict_proba(
        X_scoring
    )[:, 1]
)


scores = pd.DataFrame(
    {
        "client_id":
            scoring_df["client_id"].to_numpy(),

        "p_treatment":
            p_treatment,

        "p_control":
            p_control,

        "predicted_uplift":
            p_treatment - p_control,
    }
)


# ------------------------------------------------------------
# Score integrity checks
# ------------------------------------------------------------

assert len(scores) == 200_123
assert scores["client_id"].is_unique

assert scores[
    ["p_treatment", "p_control"]
].apply(
    lambda column: column.between(0, 1).all()
).all()

assert scores["predicted_uplift"].between(
    -1,
    1,
).all()


# Cache predictions so the scenario simulator can
# be rerun without retraining or rescoring.
PREDICTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "uplift_scoring_predictions.parquet"
)

scores.to_parquet(
    PREDICTION_PATH,
    index=False,
)


print(f"Scored customers: {len(scores):,}")

display(
    scores["predicted_uplift"].describe(
        percentiles=[
            0.01,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.99,
        ]
    )
)

Scored customers: 200,123


count    200123.000000
mean          0.032054
std           0.032954
min          -0.932151
1%           -0.020850
10%           0.000094
25%           0.009343
50%           0.025935
75%           0.046857
90%           0.072113
99%           0.144645
max           0.391554
Name: predicted_uplift, dtype: float64

## 4. Define the targeting objective

A customer with a positive predicted uplift is not necessarily worth contacting. Contacting that customer also has a cost.

I define modeled net value per contact as:

$$
\text{Modeled net value}_i
=
\widehat{\tau}_i \times V-C
$$

where \(V\) is an assumed value per incremental conversion and \(C\) is an assumed cost per contact.

The decision system contacts customers with positive modeled net value, prioritizing those with the largest predicted contribution while respecting the available budget.

I assume constant conversion value and contact cost across customers. Under those assumptions, selecting the highest positive net-value customers is sufficient for this simple budget-constrained problem.

If customer-specific contact costs, channel constraints, or other operational restrictions are introduced later, the optimization problem will need to be expanded.

**The numerical inputs below are illustrative, not facts supplied by X5.**


In [4]:

# ============================================================
# 4. First budget-constrained contact policy
#
# These are HYPOTHETICAL business inputs.
# They are not observed costs or revenues from X5.
# ============================================================

CONVERSION_VALUE = 20.0

CONTACT_COST = 0.25

BUDGET = 10_000.0


# Build a complete candidate ranking and contact decision.
decisions = build_contact_policy(
    scores=scores,
    conversion_value=CONVERSION_VALUE,
    contact_cost=CONTACT_COST,
    budget=BUDGET,
)


# ------------------------------------------------------------
# Summarize the selected customer population.
# ------------------------------------------------------------

selected = decisions.loc[
    decisions["contact"]
]

selected_count = len(selected)

contact_spend = (
    selected_count * CONTACT_COST
)

modeled_incremental_conversions = (
    selected["predicted_uplift"].sum()
)

modeled_incremental_value = (
    modeled_incremental_conversions
    * CONVERSION_VALUE
)

modeled_net_value = (
    modeled_incremental_value
    - contact_spend
)


print(f"Selected customers: {selected_count:,}")

print(
    f"Selected population: "
    f"{selected_count / len(scores):.2%}"
)

print(
    f"Contact spend: "
    f"{contact_spend:,.2f} monetary units"
)

print(
    f"Modeled incremental conversions: "
    f"{modeled_incremental_conversions:,.2f}"
)

print(
    f"Modeled net value: "
    f"{modeled_net_value:,.2f} monetary units"
)


# ------------------------------------------------------------
# Confirm the policy is economically feasible.
# ------------------------------------------------------------

assert contact_spend <= BUDGET + 1e-8

assert (
    selected["modeled_net_value_per_contact"] > 0
).all()

display(
    decisions.head(10)
)

Selected customers: 40,000
Selected population: 19.99%
Contact spend: 10,000.00 monetary units
Modeled incremental conversions: 3,291.91
Modeled net value: 55,838.15 monetary units


,client_id,predicted_uplift,p_treatment,modeled_incremental_value,modeled_net_value_per_contact,contact
0,f4d38daf8d,0.391554,0.814175,7.831082,7.581082,True
1,ce88b9b2a9,0.345603,0.598421,6.912067,6.662067,True
2,fb433c8b4a,0.343616,0.566880,6.872316,6.622316,True
3,c8711e609c,0.338616,0.600123,6.772313,6.522313,True
4,41916d4380,0.327776,0.568821,6.555526,6.305526,True
5,3f2a1ee269,0.324031,0.557174,6.480628,6.230628,True
6,2533380e77,0.322575,0.536670,6.451494,6.201494,True
7,f32b09abe7,0.318014,0.507261,6.360283,6.110283,True
8,102b63dc50,0.318010,0.537046,6.360198,6.110198,True
9,c8a84c2fa3,0.312220,0.544884,6.244391,5.994391,True


## 5. Compare alternative targeting policies

I compare the budget-constrained uplift policy against four reference strategies.

**Random targeting:** Randomly select the same number of customers as the uplift policy.

**Purchase-propensity targeting:** Contact the same number of customers, prioritizing those with the highest predicted purchase probability under treatment. This tests whether simply targeting likely purchasers is sufficient.

**Contact everyone:** Contact the entire scoring population, regardless of budget or predicted uplift. This is an unconstrained reference and may not be financially feasible.

**Contact nobody:** Spend nothing and generate no modeled incremental conversions.

All policy comparisons in this section use the same fitted uplift model to estimate their outcomes. Therefore, they demonstrate how the model translates into different decisions; they do not independently establish which policy would generate the most real-world conversions.

In [5]:
# ============================================================
# 5. Compare targeting policies
#
# The budget-constrained, random, and purchase-propensity
# policies contact the SAME number of customers.
#
# This separates differences in customer selection from
# differences caused simply by contacting more people.
# ============================================================

optimized_ids = (
    decisions.loc[
        decisions["contact"],
        "client_id",
    ]
)

n_contacts = len(optimized_ids)


# ------------------------------------------------------------
# Random baseline: same number of contacts.
# ------------------------------------------------------------

random_ids = (
    scores.sample(
        n=n_contacts,
        random_state=42,
    )["client_id"]
)


# ------------------------------------------------------------
# Purchase-propensity baseline:
#
# Rank by P(Y=1 | X, treatment=1) rather than by uplift.
#
# This prioritizes likely purchasers, including customers
# who might have purchased without any contact.
# ------------------------------------------------------------

purchase_propensity_ids = (
    scores
    .sort_values(
        [
            "p_treatment",
            "client_id",
        ],
        ascending=[False, True],
    )
    .head(n_contacts)["client_id"]
)


# ------------------------------------------------------------
# Build a policy comparison using one shared function.
# ------------------------------------------------------------

POLICIES = {
    "Uplift optimized":
        optimized_ids,

    "Random — same contacts":
        random_ids,

    "High purchase propensity":
        purchase_propensity_ids,

    "Contact everyone":
        scores["client_id"],

    "Contact nobody":
        [],
}


policy_rows = []

for policy_name, selected_ids in POLICIES.items():

    policy_rows.append(
        summarize_policy(
            scores=scores,
            selected_client_ids=selected_ids,
            conversion_value=CONVERSION_VALUE,
            contact_cost=CONTACT_COST,
            budget=BUDGET,
            policy_name=policy_name,
        )
    )


policy_comparison = (
    pd.DataFrame(policy_rows)
    .set_index("policy")
)

display(
    policy_comparison.round(2)
)

,contacts,contact_pct,modeled_incremental_conversions,modeled_incremental_value,contact_spend,modeled_net_value,within_budget
policy,,,,,,,
Uplift optimized,40000,19.99,3291.91,65838.15,10000.00,55838.15,True
Random — same contacts,40000,19.99,1285.19,25703.73,10000.00,15703.73,True
High purchase propensity,40000,19.99,482.64,9652.77,10000.00,-347.23,True
Contact everyone,200123,100.00,6414.77,128295.43,50030.75,78264.68,False
Contact nobody,0,0.00,0.00,0.00,0.00,0.00,True


## 6. Budget sensitivity analysis

The original X5 dataset does not specify how much the business can spend on communication.

Instead of choosing one budget and treating it as a fact, I evaluate several hypothetical budgets while holding conversion value and contact cost constant.

This allows me to examine how the selected population, modeled incremental conversions, and modeled net value change as the contact budget expands.

The objective is not necessarily to use the entire budget. If the model predicts that additional contacts have zero or negative net value, the decision engine should stop selecting customers even when money remains available.

In [6]:
# ============================================================
# 6. Budget sensitivity
#
# Reuse the same scores and economic assumptions.
# Only the contact budget changes.
#
# No model retraining is necessary.
# ============================================================

BUDGET_SCENARIOS = [
    0,
    2_500,
    5_000,
    10_000,
    20_000,
]


budget_rows = []

for scenario_budget in BUDGET_SCENARIOS:

    scenario_decisions = build_contact_policy(
        scores=scores,
        conversion_value=CONVERSION_VALUE,
        contact_cost=CONTACT_COST,
        budget=scenario_budget,
    )

    selected_ids = (
        scenario_decisions.loc[
            scenario_decisions["contact"],
            "client_id",
        ]
    )

    summary = summarize_policy(
        scores=scores,
        selected_client_ids=selected_ids,
        conversion_value=CONVERSION_VALUE,
        contact_cost=CONTACT_COST,
        budget=scenario_budget,
        policy_name="Uplift optimized",
    )

    budget_rows.append(
        {
            "budget": scenario_budget,
            **summary,
        }
    )


budget_sensitivity = (
    pd.DataFrame(budget_rows)
    .set_index("budget")
)

display(
    budget_sensitivity.round(2)
)

,policy,contacts,contact_pct,modeled_incremental_conversions,modeled_incremental_value,contact_spend,modeled_net_value,within_budget
budget,,,,,,,,
0,Uplift optimized,0,0.00,0.00,0.00,0.0,0.00,True
2500,Uplift optimized,10000,5.00,1253.93,25078.57,2500.0,22578.57,True
5000,Uplift optimized,20000,9.99,2066.17,41323.31,5000.0,36323.31,True
10000,Uplift optimized,40000,19.99,3291.91,65838.15,10000.0,55838.15,True
20000,Uplift optimized,80000,39.98,4980.83,99616.53,20000.0,79616.53,True


## 7. Conversion-value sensitivity

The modeled value of a contact depends on the assumed value of an incremental conversion.

A customer with small positive uplift might not justify contact when conversions are worth relatively little, but could become economically eligible when their value is higher.

I therefore vary conversion value while holding contact cost and budget fixed.

This demonstrates how a business assumption changes the targeting policy without changing the underlying ML predictions.

The analysis remains hypothetical because X5 does not provide a verified monetary value per post-communication conversion.

In [7]:
# ============================================================
# 7. Conversion-value sensitivity
#
# Keep cost and budget fixed.
# Vary the assumed value of one incremental conversion.
# ============================================================

CONVERSION_VALUE_SCENARIOS = [
    5.0,
    10.0,
    20.0,
    40.0,
]


value_rows = []

for scenario_value in CONVERSION_VALUE_SCENARIOS:

    scenario_decisions = build_contact_policy(
        scores=scores,
        conversion_value=scenario_value,
        contact_cost=CONTACT_COST,
        budget=BUDGET,
    )

    selected_ids = (
        scenario_decisions.loc[
            scenario_decisions["contact"],
            "client_id",
        ]
    )

    summary = summarize_policy(
        scores=scores,
        selected_client_ids=selected_ids,
        conversion_value=scenario_value,
        contact_cost=CONTACT_COST,
        budget=BUDGET,
        policy_name="Uplift optimized",
    )

    value_rows.append(
        {
            "conversion_value": scenario_value,
            **summary,
        }
    )


value_sensitivity = (
    pd.DataFrame(value_rows)
    .set_index("conversion_value")
)

display(
    value_sensitivity.round(2)
)

,policy,contacts,contact_pct,modeled_incremental_conversions,modeled_incremental_value,contact_spend,modeled_net_value,within_budget
conversion_value,,,,,,,,
5.0,Uplift optimized,40000,19.99,3291.91,16459.54,10000.0,6459.54,True
10.0,Uplift optimized,40000,19.99,3291.91,32919.08,10000.0,22919.08,True
20.0,Uplift optimized,40000,19.99,3291.91,65838.15,10000.0,55838.15,True
40.0,Uplift optimized,40000,19.99,3291.91,131676.31,10000.0,121676.31,True


## 8. Conclusions and limitations

I converted the provisional logistic T-learner into a budget-constrained customer targeting system. I refitted the model on all 200,039 labeled customers and generated uplift predictions for the 200,123 customers in the unlabeled X5 scoring population.

The decision engine combines predicted uplift with an assumed conversion value, contact cost, and budget to select customers with positive modeled net value. I compared this policy against random targeting, purchase-propensity targeting, contacting everyone, and contacting nobody.

### Key findings

Under the main hypothetical scenario, I assumed a conversion value of 20 monetary units, a contact cost of 0.25 monetary units, and a total budget of 10,000 monetary units.

- Customers scored: **200,123**
- Customers selected: **40,000 (19.99% of the scoring population)**
- Assumed contact budget: **10,000 monetary units**
- Modeled incremental conversions: **3,291.91**
- Modeled incremental value: **65,838.15 monetary units**
- Contact spend: **10,000 monetary units**
- Modeled net value: **55,838.15 monetary units**

### Policy comparison

With the same 40,000-contact limit, random targeting produced 1,285.19 modeled incremental conversions and 15,703.73 monetary units in modeled net value. Targeting customers with the highest predicted purchase probability produced 482.64 modeled incremental conversions and a modeled net value of −347.23 monetary units.

This illustrates why predicting who will purchase is different from predicting whose purchasing probability may increase because of treatment.

Contacting everyone produced a modeled net value of 78,264.68 monetary units, but required 50,030.75 monetary units in contact spending. It therefore exceeded the main scenario's budget and was not a feasible alternative.

### Sensitivity analysis

As the assumed budget increased from 2,500 to 20,000 monetary units, the optimized policy expanded from 10,000 to 80,000 contacts. Modeled incremental conversions increased from 1,253.93 to 4,980.83, while modeled net value increased from 22,578.57 to 79,616.53 monetary units.

I also varied the assumed value per incremental conversion from 5 to 40 monetary units while keeping the budget and contact cost fixed. Modeled net value ranged from 6,459.54 to 121,676.31 monetary units. The policy continued selecting 40,000 customers in each scenario because the budget remained binding and the selected contacts had positive modeled net value.

All four decision-engine unit tests passed, confirming the expected selection, budget, and reconciliation behavior on small examples with known answers.

### Limitations

These findings are model-based scenario estimates, not realized campaign results. The X5 scoring population has no observed treatment outcomes, and the dataset does not provide verified conversion values, contact costs, or monetary units.

The policy comparisons are also evaluated using predictions from the same model that generates the uplift-based ranking. They demonstrate how the decision logic behaves, but they do not independently establish that uplift targeting would outperform the alternative policies in a real campaign. The random-targeting comparison uses one random sample rather than an average across repeated samples.

The available X5 documentation does not establish randomized treatment assignment. Interpreting the predicted differences as causal effects therefore requires assumptions that cannot be fully verified from the dataset.

Finally, the decision engine assumes one contact per customer, constant conversion value, constant contact cost, and no additional eligibility or channel constraints. A future version could incorporate customer-specific economics, multiple communication channels, contact-frequency limits, and other operational restrictions.

The next stage is to improve reproducibility through experiment tracking, model versioning, and artifact management.